In [1]:
import folium
import pandas as pd
import geopandas as gpd
import requests

In [2]:
# Load Irish county boundaries from Tailte Eireann via data.gov.ie
# Source: https://data.gov.ie/dataset/counties-national-statutory-boundaries-20191

counties = gpd.read_file("data/counties.geojson")

In [3]:
# Check the data loaded correctly
print(counties.head()) #Print the first few rows of counties.geojson variable
print(counties.columns) #Print column titles from counties.geojson variable

                  ENGLISH GAEILGE             CONTAE   COUNTY  PROVINCE  \
0     DUBLIN CITY COUNCIL    None  Baile Átha Cliath   DUBLIN  Leinster   
1       CORK CITY COUNCIL    None           Corcaigh     CORK   Munster   
2     GALWAY CITY COUNCIL    None           Gaillimh   GALWAY  Connacht   
3   OFFALY COUNTY COUNCIL    None        Uíbh Fhailí   OFFALY  Leinster   
4  WICKLOW COUNTY COUNCIL    None      Cill Mhantáin  WICKLOW  Leinster   

                                   GUID  CENTROID_X  CENTROID_Y          AREA  \
0  2ae19629-1433-13a3-e055-000000000001   716469.75   735272.06  1.283502e+08   
1  2ae19629-1434-13a3-e055-000000000001   565833.13   571933.83  1.865976e+08   
2  2ae19629-1435-13a3-e055-000000000001   530067.66   726500.52  5.069505e+07   
3  2ae19629-1496-13a3-e055-000000000001   631261.72   709672.35  2.000025e+09   
4  2ae19629-149e-13a3-e055-000000000001   707784.79   690738.10  2.025161e+09   

    CC_ID  ESRI_OID                                           

In [4]:
# Load CSO afforestation data by county
# Source: Central Statistics Office Ireland AFA01 Afforestation Area 2023
# https://www.cso.ie

forestry = pd.read_csv("data/forestry.csv") #Create DataFrame 'forestry'

# Check the data loaded correctly
print(forestry.head())
print(forestry.columns)

      Statistic Label  Year      County              Species  \
0  Afforestation Area  2023  Co. Carlow  Total Afforestation   
1  Afforestation Area  2023  Co. Carlow  Total Afforestation   
2  Afforestation Area  2023  Co. Carlow  Total Afforestation   
3  Afforestation Area  2023  Co. Carlow  Total Afforestation   
4  Afforestation Area  2023  Co. Carlow            Broadleaf   

          Forest Owner      UNIT  VALUE  
0  Total Afforestation  Hectares   24.0  
1               Farmer  Hectares    4.0  
2           Non-Farmer  Hectares   19.0  
3        Public Sector  Hectares    NaN  
4  Total Afforestation  Hectares   10.0  
Index(['Statistic Label', 'Year', 'County', 'Species', 'Forest Owner', 'UNIT',
       'VALUE'],
      dtype='str')


In [5]:
# Check unique county names in forestry data
print(forestry['County'].unique())
print(len(forestry['County'].unique()))

<StringArray>
[   'Co. Carlow',     'Co. Cavan',     'Co. Clare',      'Co. Cork',
   'Co. Donegal',    'Co. Dublin',    'Co. Galway',     'Co. Kerry',
   'Co. Kildare',  'Co. Kilkenny',     'Co. Laois',   'Co. Leitrim',
  'Co. Limerick',  'Co. Longford',     'Co. Louth',      'Co. Mayo',
     'Co. Meath',  'Co. Monaghan',    'Co. Offaly', 'Co. Roscommon',
     'Co. Sligo', 'Co. Tipperary', 'Co. Waterford', 'Co. Westmeath',
   'Co. Wexford',   'Co. Wicklow',       'Ireland']
Length: 27, dtype: str
27


In [8]:
print(forestry['Species'].unique())

<StringArray>
['Total Afforestation', 'Broadleaf', 'Conifer']
Length: 3, dtype: str


In [9]:
print(forestry['Forest Owner'].unique())

<StringArray>
['Total Afforestation', 'Farmer', 'Non-Farmer', 'Public Sector']
Length: 4, dtype: str


In [10]:
# Remove Ireland row
df_filtered = forestry[forestry['County'] != 'Ireland']
print("After removing Ireland:", len(df_filtered['County'].unique()), "counties")

After removing Ireland: 26 counties


In [12]:
# Filter counties by species and owner
df_filtered = df_filtered[(df_filtered['Species'] == 'Total Afforestation') & 
                          (df_filtered['Forest Owner'] == 'Total Afforestation')]
print(df_filtered[['County', 'VALUE']])

            County  VALUE
0       Co. Carlow   24.0
12       Co. Cavan   77.0
24       Co. Clare  117.0
36        Co. Cork  127.0
48     Co. Donegal   21.0
60      Co. Dublin    NaN
72      Co. Galway  139.0
84       Co. Kerry   98.0
96     Co. Kildare   28.0
108   Co. Kilkenny   59.0
120      Co. Laois   21.0
132    Co. Leitrim  102.0
144   Co. Limerick   29.0
156   Co. Longford   64.0
168      Co. Louth   34.0
180       Co. Mayo  131.0
192      Co. Meath   54.0
204   Co. Monaghan   10.0
216     Co. Offaly   31.0
228  Co. Roscommon  189.0
240      Co. Sligo   49.0
252  Co. Tipperary   98.0
264  Co. Waterford   19.0
276  Co. Westmeath   76.0
288    Co. Wexford   14.0
300    Co. Wicklow   41.0


In [15]:
#Replace NaN values with 0
df_filtered['VALUE'] = df_filtered['VALUE'].fillna(0)
print(df_filtered[['County', 'VALUE']])

            County  VALUE
0       Co. Carlow   24.0
12       Co. Cavan   77.0
24       Co. Clare  117.0
36        Co. Cork  127.0
48     Co. Donegal   21.0
60      Co. Dublin    0.0
72      Co. Galway  139.0
84       Co. Kerry   98.0
96     Co. Kildare   28.0
108   Co. Kilkenny   59.0
120      Co. Laois   21.0
132    Co. Leitrim  102.0
144   Co. Limerick   29.0
156   Co. Longford   64.0
168      Co. Louth   34.0
180       Co. Mayo  131.0
192      Co. Meath   54.0
204   Co. Monaghan   10.0
216     Co. Offaly   31.0
228  Co. Roscommon  189.0
240      Co. Sligo   49.0
252  Co. Tipperary   98.0
264  Co. Waterford   19.0
276  Co. Westmeath   76.0
288    Co. Wexford   14.0
300    Co. Wicklow   41.0


In [16]:
def prepare_forestry_data(df, species, owner='Total Afforestation'):
 
    # Remove Ireland row
    df = df[df['County'] != 'Ireland']
    
    # Filter counties by species and owner
    df = df[(df['Species'] == species) & 
            (df['Forest Owner'] == owner)]
    
    # Clean county names to match GeoJSON format
    df = df.copy()
    df['County'] = df['County'].str.replace('Co. ', '').str.upper()
    
    # Replace NaN values with 0
    df['VALUE'] = df['VALUE'].fillna(0)
    
    return df

In [21]:
print(total['County'].unique())

<StringArray>
[   'CARLOW',     'CAVAN',     'CLARE',      'CORK',   'DONEGAL',    'DUBLIN',
    'GALWAY',     'KERRY',   'KILDARE',  'KILKENNY',     'LAOIS',   'LEITRIM',
  'LIMERICK',  'LONGFORD',     'LOUTH',      'MAYO',     'MEATH',  'MONAGHAN',
    'OFFALY', 'ROSCOMMON',     'SLIGO', 'TIPPERARY', 'WATERFORD', 'WESTMEATH',
   'WEXFORD',   'WICKLOW']
Length: 26, dtype: str


In [17]:
# Prep data for all three map layers
total = prepare_forestry_data(forestry, 'Total Afforestation')
broadleaf = prepare_forestry_data(forestry, 'Broadleaf')
conifer = prepare_forestry_data(forestry, 'Conifer')

print("Total rows:", len(total))
print("Broadleaf rows:", len(broadleaf))
print("Conifer rows:", len(conifer))

Total rows: 26
Broadleaf rows: 26
Conifer rows: 26
